In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


Mounted at /content/drive/


In [ ]:
image_dir ='/content/drive/Shared with me/xrays/'  # Update if needed
csv_path ='/content/dental_labels_final.csv'

In [ ]:
!pip install torch torchvision pandas scikit-learn matplotlib opencv-python gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)

# Base binary columns
base_binary_columns = [
    'bone_loss', 'pdl_widening', 'lamina_dura_loss',
    'periapical_lesion', 'unhealed_extraction',
    'tooth_mobility', 'diabetes_likelihood', 'sex'
]

# Tooth-specific binary columns
tooth_columns = [col for col in df.columns if col.startswith('tooth_')]

# Final binary & float labels
binary_columns = base_binary_columns + tooth_columns
float_columns = ['periodontitis_score', 'pbl_severity']


In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


In [ ]:
import os, cv2, torch
import numpy as np
from torch.utils.data import Dataset

class DentalDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['filename']
        img_path = os.path.join(self.image_dir, fname)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (224, 224)) if image is not None else np.zeros((224, 224), dtype=np.uint8)
        image = np.expand_dims(image, axis=2)
        if self.transform:
            image = self.transform(image)
        binary_target = torch.tensor(self.df.iloc[idx][binary_columns].values.astype(np.float32))
        float_target = torch.tensor(self.df.iloc[idx][float_columns].values.astype(np.float32))
        return image, binary_target, float_target


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = DentalDataset(train_df, image_dir, transform)
val_dataset = DentalDataset(val_df, image_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [ ]:
from torchvision import models
import torch.nn as nn

class MultiOutputResNet(nn.Module):
    def __init__(self, num_binary, num_float):
        super().__init__()
        self.base_model = models.resnet18(pretrained=True)
        self.base_model.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        self.base_model.fc = nn.Identity()
        self.binary_head = nn.Linear(512, num_binary)
        self.float_head = nn.Linear(512, num_float)

    def forward(self, x):
        x = self.base_model(x)
        return self.binary_head(x), self.float_head(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiOutputResNet(num_binary=len(binary_columns), num_float=len(float_columns)).to(device)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 127MB/s]


In [ ]:
import torch.optim as optim

criterion_bce = nn.BCEWithLogitsLoss()
criterion_mse = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
from tqdm import tqdm

def train_epoch(model, dataloader):
    model.train()
    total_loss = 0
    for images, binary_targets, float_targets in tqdm(dataloader):
        images, binary_targets, float_targets = images.to(device), binary_targets.to(device), float_targets.to(device)
        optimizer.zero_grad()
        binary_out, float_out = model(images)
        loss = criterion_bce(binary_out, binary_targets) + criterion_mse(float_out, float_targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def validate_epoch(model, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, binary_targets, float_targets in tqdm(dataloader):
            images, binary_targets, float_targets = images.to(device), binary_targets.to(device), float_targets.to(device)
            binary_out, float_out = model(images)
            loss = criterion_bce(binary_out, binary_targets) + criterion_mse(float_out, float_targets)
            total_loss += loss.item()
    return total_loss / len(dataloader)


In [ ]:
for epoch in range(40):
    train_loss = train_epoch(model, train_loader)
    val_loss = validate_epoch(model, val_loader)
    print(f"📅 Epoch {epoch+1}/40 | 🏋️ Train Loss: {train_loss:.4f} | 🧪 Val Loss: {val_loss:.4f}")


100%|██████████| 20/20 [00:00<00:00, 23.16it/s]


📅 Epoch 1/40 | 🏋️ Train Loss: 0.3555 | 🧪 Val Loss: 0.2678


100%|██████████| 20/20 [00:00<00:00, 23.91it/s]


📅 Epoch 2/40 | 🏋️ Train Loss: 0.2817 | 🧪 Val Loss: 0.2665


100%|██████████| 20/20 [00:00<00:00, 23.33it/s]


...

📅 Epoch 38/40 | 🏋️ Train Loss: 0.2798 | 🧪 Val Loss: 0.2669


100%|██████████| 20/20 [00:00<00:00, 23.20it/s]


📅 Epoch 39/40 | 🏋️ Train Loss: 0.2804 | 🧪 Val Loss: 0.2668


100%|██████████| 20/20 [00:00<00:00, 23.01it/s]

📅 Epoch 40/40 | 🏋️ Train Loss: 0.2797 | 🧪 Val Loss: 0.2672


In [ ]:
torch.save(model.state_dict(), "model_dental.pth")


In [ ]:
def predict_from_image(img_path):
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (224, 224))
    image = np.expand_dims(image, axis=(0, 3))  # Add channel
    image = transform(image[0])  # Apply transform
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        binary_logits, float_outputs = model(image)
        binary_probs = torch.sigmoid(binary_logits).cpu().numpy().flatten()
        float_preds = float_outputs.cpu().numpy().flatten()

    # Filter out tooth-related predictions
    binary_results = {
        label: f"{'YES' if p > 0.7 else 'NO'} ({p:.2f})"
        for label, p in zip(binary_columns, binary_probs)
        if not label.startswith("tooth_")
    }

    float_results = {
        label: f"{v:.2f}" for label, v in zip(float_columns, float_preds)
    }

    return {**binary_results, **float_results}


In [ ]:
import gradio as gr

def gradio_interface(image):
    # Save uploaded image temporarily
    temp_path = "/content/temp_image.jpg"
    image.save(temp_path)
    results = predict_from_image(temp_path)
    return results

gr.Interface(
    fn=gradio_interface,
    inputs=gr.Image(type="pil", label="Upload Dental X-ray"),
    outputs=[gr.JSON(label="Prediction Results")],
    title="🦷 Dental X-ray Analysis",
    description="Predicts dental conditions and scores from an uploaded X-ray using a multi-output ResNet model."
).launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6cf9965100e3743f7e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
